In [ ]:

!pip install -qqq pypdf sentence-transformers faiss-cpu groq langchain langchain-community

In [ ]:
def extract_text_from_pdf(pdf_path):
    """Extracts text from a given PDF file."""
    text = ""
    try:
        reader = PdfReader(pdf_path)
        for page in reader.pages:
            text += page.extract_text() or ""
    except Exception as e:
        print(f"Error reading PDF {pdf_path}: {e}")
        return None
    return text


pdf_file_path = '/content/iso27001.pdf'

doc_text = extract_text_from_pdf(pdf_file_path)

if doc_text:
    print(f"Text extracted. Length: {len(doc_text)} characters.")
else:
    print(f"Failed to extract text from {pdf_file_path}. Please ensure the PDF path is correct and accessible.")

In [ ]:

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)

if doc_text:
    chunks = text_splitter.split_text(doc_text)
    print(f"Created {len(chunks)} chunks.")

    for i, chunk in enumerate(chunks[:2]):
        print(f"\n--- Chunk {i+1} ---")
        print(chunk)
else:
    chunks = []
    print("No text to chunk.")

In [ ]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

if chunks:
    print("Generating embeddings for chunks...")
    chunk_embeddings = embedding_model.encode(chunks, show_progress_bar=True)
    print(f"Generated embeddings with shape: {chunk_embeddings.shape}")
else:
    chunk_embeddings = np.array([])
    print("No chunks to embed.")

In [ ]:
if chunk_embeddings.size > 0:
    dimension = chunk_embeddings.shape[1]
    faiss_index = faiss.IndexFlatL2(dimension)
    faiss_index.add(chunk_embeddings)
    print(f"FAISS index created with {faiss_index.ntotal} vectors and dimension {dimension}.")
else:
    faiss_index = None
    print("No embeddings to add to FAISS index.")

In [ ]:
import re
def hybrid_retrieval(query, top_k_vector=5, top_k_keyword=5, score_threshold=0.7):
    """Performs hybrid retrieval combining vector search and keyword search."""
    query_embedding = embedding_model.encode([query])


    distances, faiss_indices = faiss_index.search(query_embedding, top_k_vector)
    retrieved_chunks_vector = [chunks[i] for i in faiss_indices[0]]
    vector_scores = 1 - distances[0]


    retrieved_chunks_keyword = []
    keyword_scores = []
    for i, chunk in enumerate(chunks):
        if query.lower() in chunk.lower():
            retrieved_chunks_keyword.append(chunk)
            keyword_scores.append(1.0)


    combined_results = []
    for i, chunk in enumerate(retrieved_chunks_vector):
        combined_results.append({"chunk": chunk, "score": vector_scores[i]})

    for i, chunk in enumerate(retrieved_chunks_keyword):

        if not any(res['chunk'] == chunk for res in combined_results):
             combined_results.append({"chunk": chunk, "score": keyword_scores[i]})


    combined_results.sort(key=lambda x: x['score'], reverse=True)
    final_retrieved_context = [res['chunk'] for res in combined_results if res['score'] >= score_threshold]

    return final_retrieved_context

def generate_rag_answer(user_query):
    """Generates an answer using the RAG approach with Groq LLM."""
    if not faiss_index or chunk_embeddings.size == 0:
        return "Error: Document processing not complete or no embeddings generated."


    retrieved_context = hybrid_retrieval(user_query, top_k_vector=3, top_k_keyword=2, score_threshold=0.5)

    if not retrieved_context:
        return "Information not found in the document context."

    context_str = "\n\n".join(retrieved_context)


    system_prompt = (
        "You are a highly specialized text generation model. Your sole task is to provide a concise, factual answer to the user's question, based *only* on the provided context. "
        "Your output MUST be **plain text only**. **DO NOT** include any XML tags, markdown, special characters, rationales, references, or any other formatting in your answer. "
        "If the answer cannot be found or logically derived from the context, respond strictly with 'Information not found.'. "
        "Your response will be directly inserted into an answer field, so it must contain *only* the answer text and nothing else."
    )

    user_prompt = f"""
    <context>
    {context_str}
    </context>

    Based on the above context, answer the following question: {user_query}
    """


    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {
                    "role": "system",
                    "content": system_prompt,
                },
                {
                    "role": "user",
                    "content": user_prompt,
                }
            ],
            model="llama-3.1-8b-instant",
            temperature=0.1,
            max_tokens=500,
            top_p=1,
            stop=None,
            stream=False
        )
        llm_response = chat_completion.choices[0].message.content
    except Exception as e:
        return f"Error calling Groq API: {e}"


    llm_response_cleaned = re.sub(r'<[^>]+>', '', llm_response).strip()


    formatted_output = f"""
   Topic: Document Knowledge Extraction:

Question:{user_query}


Answer: {llm_response_cleaned}
"""

    return formatted_output

print("RAG generation function defined.")

In [ ]:

user_query_1 = " scope of the information security management system?"
print(f"\nQuery 1: {user_query_1}")
response_1 = generate_rag_answer(user_query_1)
print(response_1)

user_query_2 = "Explain deep learning and large language models."
print(f"\nQuery 2: {user_query_2}")
response_2 = generate_rag_answer(user_query_2)
print(response_2)

user_query_3 = "What is the capital of France?"
print(f"\nQuery 3: {user_query_3}")
response_3 = generate_rag_answer(user_query_3)
print(response_3)

user_query_4 = "Tell me about quantum computing."
print(f"\nQuery 4: {user_query_4}")
response_4 = generate_rag_answer(user_query_4)
print(response_4)